## Develop Model — v3 (full dataset, configurable size)

In [ ]:
import sys, os
# Add project root to path so 'src' is importable
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
# Load libraries and modules
import json
import random
from pathlib import Path

from src.dataset import VizWizBinaryDataset, build_vocab
from src.model import VizWizBinaryClassifier   # updated model

from torchvision import transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn

In [ ]:
# ----- Dataset size limits -----
# Assignment cap: training data must not exceed 10 000 samples.
# Set to None to use the full split.
MAX_TRAIN_SAMPLES = 10_000
MAX_VAL_SAMPLES   = None   # use full val set

# ----- Paths -----
TRAIN_IMAGE_DIR = Path("../../data/train")
VAL_IMAGE_DIR   = Path("../../data/val")
TRAIN_ANN_PATH  = Path("../../data/Annotations/train.json")
VAL_ANN_PATH    = Path("../../data/Annotations/val.json")

# Load train annotations, filter to locally available images
with open(TRAIN_ANN_PATH) as f:
    all_train = json.load(f)

train_available = {p.name for p in TRAIN_IMAGE_DIR.glob("*.jpg")}
train_annotations = [a for a in all_train if a["image"] in train_available]
if MAX_TRAIN_SAMPLES is not None:
    train_annotations = train_annotations[:MAX_TRAIN_SAMPLES]

# Load val annotations, filter to locally available images
with open(VAL_ANN_PATH) as f:
    all_val = json.load(f)

val_available = {p.name for p in VAL_IMAGE_DIR.glob("*.jpg")}
val_annotations = [a for a in all_val if a["image"] in val_available]
if MAX_VAL_SAMPLES is not None:
    val_annotations = val_annotations[:MAX_VAL_SAMPLES]

print(f"Train annotations : {len(train_annotations)}")
print(f"Val annotations   : {len(val_annotations)}")

In [ ]:
# -------------------------------------------------------
# Transforms
# -------------------------------------------------------

# Training: augmentation + ImageNet normalization.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# Validation / test: only resize + normalize, no random augmentation.
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# -------------------------------------------------------
# Vocabulary  (built only from training questions to avoid data leakage)
# -------------------------------------------------------
questions = [a["question"] for a in train_annotations]
vocab = build_vocab(questions, min_freq=1)
MAX_LEN = 20
print(f"Vocab size : {len(vocab)}")

# -------------------------------------------------------
# Datasets
# -------------------------------------------------------
train_dataset = VizWizBinaryDataset(
    annotations=train_annotations,
    image_dir=TRAIN_IMAGE_DIR,
    vocab=vocab,
    max_len=MAX_LEN,
    transform=train_transform,
)

val_dataset = VizWizBinaryDataset(
    annotations=val_annotations,
    image_dir=VAL_IMAGE_DIR,
    vocab=vocab,
    max_len=MAX_LEN,
    transform=val_transform,
)

# -------------------------------------------------------
# DataLoaders
# -------------------------------------------------------
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches : {len(train_loader)} | Val batches : {len(val_loader)}")

In [ ]:

# -------------------------------------------------------
# Model, Loss, Optimizer, Scheduler
# -------------------------------------------------------

EMBED_DIM  = 256    # shared dimensionality for image + text features
NUM_HEADS  = 4      # attention heads (embed_dim must be divisible by num_heads)
NUM_LAYERS = 2      # number of Transformer encoder layers in the text encoder
DROPOUT    = 0.3    # dropout rate used in all sub-modules
NUM_EPOCHS = 20

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Training on: {DEVICE}")

model = VizWizBinaryClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    max_len=MAX_LEN,
    dropout=DROPOUT,
).to(DEVICE)

# --- Class imbalance: pos_weight = #negatives / #positives ---
# BCEWithLogitsLoss multiplies the loss for positive samples by this factor,
# counteracting the dataset skew so the model doesn't just predict "answerable".
num_pos = sum(int(a["answerable"]) for a in train_annotations)
num_neg = len(train_annotations) - num_pos
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(DEVICE)
print(f"Class balance — pos: {num_pos}, neg: {num_neg}, pos_weight: {pos_weight.item():.3f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# -------------------------------------------------------
# Training Loop
# -------------------------------------------------------

best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):

    # ---- Training phase ----
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        images = batch["image"].to(DEVICE)
        tokens = batch["tokens"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(images, tokens)          # [B, 1]
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Step the LR scheduler once per epoch (cosine decay).
    scheduler.step()

    # ---- Validation phase ----
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(DEVICE)
            tokens = batch["tokens"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits = model(images, tokens)
            # Convert raw logits -> probabilities -> binary predictions
            preds  = (logits.sigmoid() >= 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_acc  = correct / total
    avg_loss = train_loss / len(train_loader)

    # Save the checkpoint with the highest validation accuracy
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"LR: {scheduler.get_last_lr()[0]:.6f}"
        + (" <- best" if is_best else "")
    )

In [ ]:
# -------------------------------------------------------
# Final Evaluation on Validation Set (best checkpoint)
# -------------------------------------------------------

# Load the best weights saved during training
model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))
model.eval()

correct = total = 0
tp = tn = fp = fn = 0

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(DEVICE)
        tokens = batch["tokens"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        preds = (model(images, tokens).sigmoid() >= 0.5).float()

        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        # Confusion matrix components (for the accuracy formula in the assignment)
        tp += ((preds == 1) & (labels == 1)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

accuracy = (tp + tn) / (tp + tn + fp + fn)
print(f"Val accuracy (best checkpoint) : {accuracy:.4f}")
print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
print(f"Best val acc seen during training: {best_val_acc:.4f}")